In [0]:
# ============================================================
# GOLD LAYER - CONFIGURATION
# ============================================================

CATALOG = "fhir_assignment"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SILVER_TABLES = {
    "patient": f"{CATALOG}.{SILVER_SCHEMA}.patient",
    "encounter": f"{CATALOG}.{SILVER_SCHEMA}.encounter",
    "observation": f"{CATALOG}.{SILVER_SCHEMA}.observation",
    "condition": f"{CATALOG}.{SILVER_SCHEMA}.condition"
}

GOLD_TABLES = {
    "patient": f"{CATALOG}.{GOLD_SCHEMA}.dim_patient",
    "encounter": f"{CATALOG}.{GOLD_SCHEMA}.fact_encounter",
    "observation": f"{CATALOG}.{GOLD_SCHEMA}.fact_observation",
    "condition": f"{CATALOG}.{GOLD_SCHEMA}.fact_condition"
}


In [0]:
# ============================================================
# LOAD SILVER TABLES
# ============================================================

patient_silver = spark.table(SILVER_TABLES["patient"])
encounter_silver = spark.table(SILVER_TABLES["encounter"])
observation_silver = spark.table(SILVER_TABLES["observation"])
condition_silver = spark.table(SILVER_TABLES["condition"])


In [0]:
# ============================================================
# BUILD GOLD DATASETS
# ============================================================

from pyspark.sql.functions import (
    col,
    concat_ws,
    coalesce,
    lit,
    to_date,
    round,
    when,
    trim
)


# ============================================================
# DIMENSION: PATIENT
# ============================================================

dim_patient = (
    patient_silver
    .select(
        "patient_id",
        "gender",
        "birth_date",
        "family_name",
        "given_name",
        "name_text",
        "identifier_value",
        "city",
        "state",
        "postal_code",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "patient_name",
        coalesce(
            when(
                trim(
                    concat_ws(
                        " ",
                        col("given_name"),
                        col("family_name")
                    )
                ) != "",
                trim(
                    concat_ws(
                        " ",
                        col("given_name"),
                        col("family_name")
                    )
                )
            ),
            col("name_text")
        )
    )
    .select(
        "patient_id",
        "patient_name",
        "gender",
        "birth_date",
        "identifier_value",
        "city",
        "state",
        "postal_code",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# FACT: ENCOUNTER
# ============================================================

fact_encounter = (
    encounter_silver
    .select(
        "encounter_id",
        "patient_id",
        "status",
        "class_code",
        "class_display",
        "period_start",
        "period_end",
        "practitioner_id",
        "practitioner_name",
        "service_provider",
        "facility_id",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "encounter_date",
        to_date(col("period_start"))
    )
    .withColumn(
        "duration_hours",
        when(
            col("period_start").isNotNull()
            & col("period_end").isNotNull(),
            round(
                (
                    col("period_end").cast("long")
                    - col("period_start").cast("long")
                ) / 3600.0,
                2
            )
        )
    )
    .select(
        "encounter_id",
        "patient_id",
        "encounter_date",
        "status",
        "class_code",
        "class_display",
        "period_start",
        "period_end",
        "duration_hours",
        "practitioner_id",
        "practitioner_name",
        "service_provider",
        "facility_id",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# FACT: OBSERVATION
# ============================================================

fact_observation = (
    observation_silver
    .select(
        "observation_id",
        "patient_id",
        "status",
        "category_code",
        "observation_code",
        "observation_display",
        "effective_datetime",
        "value_quantity",
        "value_unit",
        "value_string",
        "value_codeable_text",
        "source_file",
        "ingestion_timestamp"
    )
    .withColumn(
        "observation_date",
        to_date(col("effective_datetime"))
    )
    .withColumn(
        "value_type",
        when(
            col("value_quantity").isNotNull(),
            lit("Quantity")
        )
        .when(
            col("value_string").isNotNull(),
            lit("String")
        )
        .when(
            col("value_codeable_text").isNotNull(),
            lit("CodeableConcept")
        )
        .otherwise(
            lit("Unknown")
        )
    )
    .select(
        "observation_id",
        "patient_id",
        "observation_date",
        "effective_datetime",
        "status",
        "category_code",
        "observation_code",
        "observation_display",
        "value_quantity",
        "value_unit",
        "value_string",
        "value_codeable_text",
        "value_type",
        "source_file",
        "ingestion_timestamp"
    )
)


# ============================================================
# FACT: CONDITION
# ============================================================

fact_condition = (
    condition_silver
    .select(
        "condition_id",
        "patient_id",
        "encounter_id",
        "clinical_status",
        "condition_code",
        "condition_display",
        "condition_text",
        "onset_date",
        "source_file",
        "ingestion_timestamp"
    )
)


GOLD TRANSFORMATION SANITY CHECK


print("DIM PATIENT")
display(dim_patient.limit(10))

print("FACT ENCOUNTER")
display(fact_encounter.limit(10))

print("FACT OBSERVATION")
display(fact_observation.limit(10))

print("FACT CONDITION")
display(fact_condition.limit(10))

executed for sanity check not required in the final production version notebook.

In [0]:
# ============================================================
# WRITE GOLD TABLES - INCREMENTAL MERGE
# ============================================================

from delta.tables import DeltaTable

gold_tables = {
    "patient": dim_patient,
    "encounter": fact_encounter,
    "observation": fact_observation,
    "condition": fact_condition
}

gold_keys = {
    "patient": "patient_id",
    "encounter": "encounter_id",
    "observation": "observation_id",
    "condition": "condition_id"
}

for table_name, gold_df in gold_tables.items():

    full_table_name = GOLD_TABLES[table_name]
    merge_key = gold_keys[table_name]

    target = DeltaTable.forName(
        spark,
        full_table_name
    )

    (
        target.alias("target")
        .merge(
            gold_df.alias("source"),
            f"target.{merge_key} = source.{merge_key}"
        )
        .whenMatchedUpdate(
            set={
                column: f"source.{column}"
                for column in gold_df.columns
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )